In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Fabric Metadata & Data Quality Assessment Framework (CI/CD Ready)\n",
        "\n",
        "## 1. Overview & Architecture\n",
        "This framework provides an automated, end-to-end data quality assessment for Microsoft Fabric Lakehouses. It extracts structural metadata, profiles columns, evaluates custom quality rules, generates an aggregate quality index, persists execution metrics to a Delta audit table, and evaluates deployment gates for DataOps/CI/CD pipelines.\n",
        "\n",
        "### Platform Placement\n",
        "```\n",
        "             Microsoft Fabric\n",
        "                    │\n",
        "                 OneLake\n",
        "                    │\n",
        "                 Lakehouse\n",
        "                    │\n",
        "             ┌──────┴──────┐\n",
        "             │             │\n",
        "          Metadata      Dataset\n",
        "             │             │\n",
        "             └──────┬──────┘\n",
        "                    │\n",
        "             Data Profiling\n",
        "                    │\n",
        "          Data Quality Rules\n",
        "                    │\n",
        "             Quality Score\n",
        "                    │\n",
        "       ┌────────────┴────────────┐\n",
        "       │                         │\n",
        "Delta Audit Log        CI/CD Deployment Gate\n",
        " (OneLake Delta)        (Exit Status/Payload)\n",
        "```"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 2. Configuration, Parameterization & Environment Selection\n",
        "Abstracts environment details (`DEV`, `TEST`, `PROD`) and execution options using Spark parameters/widgets. Allows dynamic orchestration via Microsoft Fabric Data Factory pipelines or Azure DevOps REST API calls."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import datetime\n",
        "import json\n",
        "from pyspark.sql import SparkSession\n",
        "from pyspark.sql.functions import (\n",
        "    col, count, when, isnull, sum as _sum, avg,\n",
        "    min as _min, max as _max, lit, rand, expr, countDistinct, current_timestamp\n",
        ")\n",
        "from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType\n",
        "\n",
        "spark = SparkSession.builder.getOrCreate()\n",
        "\n",
        "# Fetch parameters from Fabric Data Factory or default to local interactive values\n",
        "try:\n",
        "    dbutils.widgets.text(\"ENVIRONMENT\", \"DEV\", \"Target Environment\")\n",
        "    dbutils.widgets.text(\"TARGET_TABLE\", \"demo_synthetic_retail\", \"Target Table Name\")\n",
        "    dbutils.widgets.text(\"USE_SYNTHETIC_DATA\", \"True\", \"Execution Mode\")\n",
        "    dbutils.widgets.text(\"PASS_THRESHOLD\", \"85.0\", \"Quality Index Threshold\")\n",
        "    dbutils.widgets.text(\"PIPELINE_RUN_ID\", \"local_run\", \"Orchestrator Run ID\")\n",
        "\n",
        "    ENV = dbutils.widgets.get(\"ENVIRONMENT\").upper()\n",
        "    TARGET_TABLE = dbutils.widgets.get(\"TARGET_TABLE\")\n",
        "    USE_SYNTHETIC_DATA = dbutils.widgets.get(\"USE_SYNTHETIC_DATA\").lower() == \"true\"\n",
        "    PASS_THRESHOLD = float(dbutils.widgets.get(\"PASS_THRESHOLD\"))\n",
        "    PIPELINE_RUN_ID = dbutils.widgets.get(\"PIPELINE_RUN_ID\")\n",
        "except Exception:\n",
        "    ENV = \"DEV\"\n",
        "    TARGET_TABLE = \"demo_synthetic_retail\"\n",
        "    USE_SYNTHETIC_DATA = True\n",
        "    PASS_THRESHOLD = 85.0\n",
        "    PIPELINE_RUN_ID = \"local_manual_run\"\n",
        "\n",
        "SYNTHETIC_ROW_COUNT = 10000\n",
        "\n",
        "# Environment-specific behavior mapping\n",
        "ENV_CONFIG = {\n",
        "    \"DEV\": {\"strict_gate\": False, \"audit_table\": \"dev_dq_audit_log\"},\n",
        "    \"TEST\": {\"strict_gate\": True,  \"audit_table\": \"test_dq_audit_log\"},\n",
        "    \"PROD\": {\"strict_gate\": True,  \"audit_table\": \"prod_dq_audit_log\"}\n",
        "}\n",
        "current_env_cfg = ENV_CONFIG.get(ENV, ENV_CONFIG[\"DEV\"])\n",
        "\n",
        "print(f\"=== [CI/CD DataOps Gate] Running DQ Engine in '{ENV}' Environment ===\")\n",
        "print(f\"Mode: {'Synthetic Data' if USE_SYNTHETIC_DATA else f'Lakehouse Table ({TARGET_TABLE})'}\")\n",
        "print(f\"Pass Threshold: {PASS_THRESHOLD}% | Strict Gate: {current_env_cfg['strict_gate']}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 3. Data Ingestion / Synthetic Data Generation\n",
        "Generates a synthetic e-commerce retail dataset containing artificial data quality anomalies (nulls, duplicates, invalid quantities, negative prices) to demonstrate framework capabilities, or loads an existing Delta table from OneLake."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "if USE_SYNTHETIC_DATA:\n",
        "    df_base = spark.range(0, SYNTHETIC_ROW_COUNT).select(\n",
        "        when(rand() < 0.014, (col(\"id\") % 100).cast(\"integer\")).otherwise(col(\"id\").cast(\"integer\")).alias(\"order_id\"),\n",
        "        when(rand() < 0.002, lit(None)).otherwise((col(\"id\") % 1000 + 100).cast(\"string\")).alias(\"customer_id\"),\n",
        "        (col(\"id\") % 500 + 1).cast(\"integer\").alias(\"product_id\"),\n",
        "        (expr(\"abs(cast(rand() * 5 as int)) + 1\")).alias(\"quantity\"),\n",
        "        when(rand() < 0.003, -10.0).otherwise(expr(\"round(rand() * 100 + 10, 2)\")).alias(\"unit_price\"),\n",
        "        expr(\"current_timestamp()\").alias(\"transaction_timestamp\")\n",
        "    )\n",
        "    df_target = df_base\n",
        "    table_name = \"demo_synthetic_retail\"\n",
        "else:\n",
        "    df_target = spark.table(TARGET_TABLE)\n",
        "    table_name = TARGET_TABLE\n",
        "\n",
        "df_target.cache()\n",
        "total_rows = df_target.count()\n",
        "print(f\"Target Dataset '{table_name}' loaded successfully. Total Rows: {total_rows:,}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 4. Metadata Discovery\n",
        "Extracts schema specifications, column physical data types, nullability properties, and table-level dimensions."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "metadata_schema = []\n",
        "for field in df_target.schema.fields:\n",
        "    metadata_schema.append({\n",
        "        \"Column Name\": field.name,\n",
        "        \"Data Type\": field.dataType.simpleString(),\n",
        "        \"Nullable\": field.nullable\n",
        "    })\n",
        "\n",
        "df_metadata = spark.createDataFrame(metadata_schema)\n",
        "print(f\"=== METADATA PROFILE: {table_name} ===\")\n",
        "df_metadata.show(truncate=False)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 5. Statistical Data Profiling\n",
        "Computes statistical profiles for every attribute, including null ratios, distinct cardinality, min/max values, and duplicate frequency."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "profile_exprs = []\n",
        "for c in df_target.columns:\n",
        "    profile_exprs.extend([\n",
        "        count(when(col(c).isNull(), 1)).alias(f\"{c}__null_cnt\"),\n",
        "        countDistinct(c).alias(f\"{c}__distinct_cnt\"),\n",
        "        _min(c).cast(\"string\").alias(f\"{c}__min_val\"),\n",
        "        _max(c).cast(\"string\").alias(f\"{c}__max_val\")\n",
        "    ])\n",
        "\n",
        "profile_row = df_target.select(profile_exprs).collect()[0]\n",
        "\n",
        "profiling_results = []\n",
        "for c in df_target.columns:\n",
        "    null_cnt = profile_row[f\"{c}__null_cnt\"]\n",
        "    profiling_results.append({\n",
        "        \"Column\": c,\n",
        "        \"Null Count\": null_cnt,\n",
        "        \"Null Percentage\": round((null_cnt / total_rows) * 100, 2),\n",
        "        \"Distinct Values\": profile_row[f\"{c}__distinct_cnt\"],\n",
        "        \"Min Value\": str(profile_row[f\"{c}__min_val\"]),\n",
        "        \"Max Value\": str(profile_row[f\"{c}__max_val\"])\n",
        "    })\n",
        "\n",
        "df_profile = spark.createDataFrame(profiling_results)\n",
        "print(\"=== DATA PROFILING SUMMARY ===\")\n",
        "df_profile.show(truncate=False)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 6. Data Quality Rules Execution\n",
        "Applies logical data quality validations across target columns evaluating key data quality dimensions: Completeness, Uniqueness, and Validity."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "dq_checks = []\n",
        "\n",
        "# Check 1: Completeness - Customer ID Nulls\n",
        "null_cust_cnt = df_target.filter(col(\"customer_id\").isNull()).count()\n",
        "null_cust_pct = round((null_cust_cnt / total_rows) * 100, 2)\n",
        "dq_checks.append({\n",
        "    \"CheckName\": \"Null customer IDs\",\n",
        "    \"Dimension\": \"Completeness\",\n",
        "    \"MetricValue\": f\"{null_cust_pct}%\",\n",
        "    \"Status\": \"PASS\" if null_cust_pct <= 0.5 else \"FAIL\",\n",
        "    \"Weight\": 20,\n",
        "    \"Passed\": 1 if null_cust_pct <= 0.5 else 0\n",
        "})\n",
        "\n",
        "# Check 2: Uniqueness - Order ID Duplicates\n",
        "distinct_orders = df_target.select(\"order_id\").distinct().count()\n",
        "dup_order_pct = round(((total_rows - distinct_orders) / total_rows) * 100, 2)\n",
        "dq_checks.append({\n",
        "    \"CheckName\": \"Duplicate orders\",\n",
        "    \"Dimension\": \"Uniqueness\",\n",
        "    \"MetricValue\": f\"{dup_order_pct}%\",\n",
        "    \"Status\": \"WARNING\" if 0.5 < dup_order_pct <= 2.0 else (\"PASS\" if dup_order_pct <= 0.5 else \"FAIL\"),\n",
        "    \"Weight\": 25,\n",
        "    \"Passed\": 0.75 if 0.5 < dup_order_pct <= 2.0 else (1 if dup_order_pct <= 0.5 else 0)\n",
        "})\n",
        "\n",
        "# Check 3: Validity - Quantity > 0\n",
        "invalid_qty_cnt = df_target.filter(col(\"quantity\") <= 0).count()\n",
        "invalid_qty_pct = round((invalid_qty_cnt / total_rows) * 100, 2)\n",
        "dq_checks.append({\n",
        "    \"CheckName\": \"Invalid quantities\",\n",
        "    \"Dimension\": \"Validity\",\n",
        "    \"MetricValue\": f\"{invalid_qty_pct}%\",\n",
        "    \"Status\": \"PASS\" if invalid_qty_pct == 0.0 else \"FAIL\",\n",
        "    \"Weight\": 25,\n",
        "    \"Passed\": 1 if invalid_qty_pct == 0.0 else 0\n",
        "})\n",
        "\n",
        "# Check 4: Validity - Unit Price > 0\n",
        "invalid_price_cnt = df_target.filter(col(\"unit_price\") <= 0).count()\n",
        "invalid_price_pct = round((invalid_price_cnt / total_rows) * 100, 2)\n",
        "dq_checks.append({\n",
        "    \"CheckName\": \"Invalid sales amount\",\n",
        "    \"Dimension\": \"Validity\",\n",
        "    \"MetricValue\": f\"{invalid_price_pct}%\",\n",
        "    \"Status\": \"PASS\" if invalid_price_pct <= 0.5 else \"FAIL\",\n",
        "    \"Weight\": 30,\n",
        "    \"Passed\": 1 if invalid_price_pct <= 0.5 else 0\n",
        "})\n",
        "\n",
        "df_dq_results = spark.createDataFrame(dq_checks)\n",
        "print(\"=== RULE EVALUATION METRICS ===\")\n",
        "df_dq_results.select(\"CheckName\", \"Dimension\", \"MetricValue\", \"Status\").show(truncate=False)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 7. Data Quality Index (DQI) Calculation\n",
        "Computes a weighted total percentage index evaluating overall dataset health."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "total_weight = sum([r[\"Weight\"] for r in dq_checks])\n",
        "weighted_passed = sum([r[\"Weight\"] * r[\"Passed\"] for r in dq_checks])\n",
        "overall_dq_score = round((weighted_passed / total_weight) * 100, 1)\n",
        "\n",
        "print(\"=\" * 45)\n",
        "print(f\"  OVERALL DATA QUALITY SCORE: {overall_dq_score}%\")\n",
        "print(\"=\" * 45)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 8. Automated Remediation Recommendations\n",
        "Provides automated next steps and remediation advice based on rule evaluation statuses."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "print(\"=== AUTOMATED REMEDIATION RECOMMENDATIONS ===\")\n",
        "for r in dq_checks:\n",
        "    if r[\"Status\"] == \"WARNING\":\n",
        "        print(f\"- [WARNING] {r['CheckName']}: Metric is {r['MetricValue']}. Apply `dropDuplicates(['order_id'])` downstream.\")\n",
        "    elif r[\"Status\"] == \"FAIL\":\n",
        "        print(f"- [ACTION REQUIRED] {r['CheckName']}: Metric is {r['MetricValue']}. Investigate source pipeline.\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 9. Audit Logging & Result Persistence (OneLake Delta)\n",
        "Persists evaluation results and execution metadata into a Delta audit log table in OneLake for DataOps monitoring and long-term trend tracking."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "audit_payload = [{\n",
        "    \"pipeline_run_id\": PIPELINE_RUN_ID,\n",
        "    \"environment\": ENV,\n",
        "    \"target_table\": table_name,\n",
        "    \"overall_dq_score\": overall_dq_score,\n",
        "    \"checks_detail\": json.dumps(dq_checks),\n",
        "    \"execution_timestamp\": datetime.datetime.utcnow().isoformat()\n",
        "}]\n",
        "\n",
        "df_audit = spark.createDataFrame(audit_payload)\n",
        "df_audit.write \\\n",
        "    .format(\"delta\") \\\n",
        "    .mode(\"append\") \\\n",
        "    .option(\"mergeSchema\", \"true\") \\\n",
        "    .saveAsTable(current_env_cfg[\"audit_table\"])\n",
        "\n",
        "print(f\"[DataOps] Results successfully logged to Delta audit table: '{current_env_cfg['audit_table']}'\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 10. CI/CD Deployment Gate & Exit Evaluation\n",
        "Evaluates the overall score against the target environment threshold (`PASS_THRESHOLD`). Returns a JSON output payload and controls notebook exit codes for Azure DevOps / Fabric pipeline release gates."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "gate_passed = overall_dq_score >= PASS_THRESHOLD\n",
        "\n",
        "gate_output = {\n",
        "    \"status\": \"SUCCESS\" if gate_passed else \"FAILED\",\n",
        "    \"dqi_score\": overall_dq_score,\n",
        "    \"threshold\": PASS_THRESHOLD,\n",
        "    \"environment\": ENV,\n",
        "    \"target_table\": table_name,\n",
        "    \"pipeline_run_id\": PIPELINE_RUN_ID\n",
        "}\n",
        "\n",
        "print(\"\\n\" + \"=\" * 50)\n",
        "print(f\" CI/CD DEPLOYMENT GATE STATUS: {gate_output['status']}\")\n",
        "print(\"=\" * 50)\n",
        "\n",
        "if not gate_passed:\n",
        "    failure_msg = f\"Deployment Gate Blocked! DQI Score ({overall_dq_score}%) < Threshold ({PASS_THRESHOLD}%).\"\n",
        "    if current_env_cfg[\"strict_gate\"]:\n",
        "        print(f\"[CI/CD GATE BLOCKED] {failure_msg}\")\n",
        "        try:\n",
        "            mssparkutils.notebook.exit(json.dumps(gate_output))\n",
        "        except NameError:\n",
        "            pass\n",
        "        raise ValueError(failure_msg)\n",
        "    else:\n",
        "        print(f\"[WARNING] {failure_msg} (Skipping exit block because strict_gate=False in {ENV})\")\n",
        "\n",
        "try:\n",
        "    mssparkutils.notebook.exit(json.dumps(gate_output))\n",
        "except NameError:\n",
        "    print(\"[INFO] Execution finished in interactive mode.\")"
      ]
    }
  ],
  "metadata": {
    "language_info": {
      "name": "python"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 2
}